# Integrating different LLMs

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

## Open AI Model (Can only invoke if we have Amount loaded)

In [ ]:
from langchain.chat_models import init_chat_model
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
model = init_chat_model("gpt-4.1")
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}}, output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001CD8F01C690>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001CD8F01CA50>, root_client=<openai.OpenAI object at 

In [5]:
# model.invoke("Hi")

## Gemini (Below approaches are applicable for Open AI and GROQ or other APIs as well)

### Approach 1 (Universal approach)

In [ ]:
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash-lite")
response = model.invoke("Hi")
response.content

'Hi there! How can I help you today?'

### Approach 2 (LLM specific API)

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
response = model.invoke("Hi")
response.content

'Hi there! How can I help you today?'

## GROQ

In [5]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("What is 2+2=?, just give straight forward answer for example what is 3+3 answer should be just 6")
response.content

'<think>\nOkay, so the user asked, "What is 2+2=?, just give straight forward answer for example what is 3+3 answer should be just 6." Alright, let me start by understanding exactly what they need.\n\nFirst, they want the answer to 2+2. The example they gave is 3+3=6, so they\'re expecting a very direct answer without any extra explanation. But maybe they want to confirm that I\'m following the same pattern. Let me check if there\'s any trick here. Sometimes people ask 2+2 to test if you know it\'s a trick question, but in most cases, it\'s straightforward. \n\nIn basic arithmetic, 2+2 is 4. That\'s the standard answer. The example they provided with 3+3=6 is correct, so they probably just want the same format for 2+2. Let me make sure there\'s no context where 2+2 might not be 4, like in some special mathematical context or wordplay. For instance, in base 3, 2+2 would be 11, but that\'s probably not what they\'re asking here. The example uses 3+3=6, which is base 10, so the user is li

In [6]:
from langchain_groq import ChatGroq
model = ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("Hi")
response.content

'<think>\nOkay, the user said "Hi". That\'s a greeting. I should respond in a friendly and welcoming way. Let me make sure to acknowledge their greeting and offer assistance. Maybe something like, "Hello! How can I assist you today?" That\'s open-ended and invites them to ask questions or share what they need help with. I need to keep it simple and approachable. Alright, that should work.\n</think>\n\nHello! How can I assist you today?'

## OLLAMA Local Models

In [8]:
from langchain.chat_models import init_chat_model
model = init_chat_model("ollama:gemma4:12b-it-qat")
response = model.invoke("What is 2+2=?, just give straight forward answer for example what is 3+3 answer should be just 6")
response.content

'4'

In [7]:
from langchain_ollama import ChatOllama

# Initialize the ChatOllama model directly
model = ChatOllama(
    model="gemma4:12b-it-qat",              # Replace with the model you have pulled
    temperature=0                # Optional: customize generation parameters
)

response = model.invoke("Hi")
response.content


'Hello! How can I help you today?'

# Streaming and Batch

## Streaming
Most of the models can stream, i.e., as model streams the output tok by token we can display it progressively instead of displaying entire response at a time, this will improve User Experience, basically it yields the output chunks as they are produced by the model

In [16]:
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash")
for chunk in model.stream("write a 200 words paragraph on LangChain"):
    print(chunk.text)

LangChain is an open-source framework designed to simplify the development of sophisticated applications powered by large language models (LLMs). It acts as a bridge, connecting LLMs with external data
 sources and computational tools, thereby overcoming the inherent limitations of models confined to their training data. At its core, LangChain enables developers to "chain" together various components, facilitating more complex and intelligent behaviors than a standalone LLM can achieve.
 Key features include flexible interfaces for diverse LLMs, robust prompt management, and the ability to define sequences of operations called "Chains." More advanced functionalities come with "Agents," which empower LLMs to make dynamic decisions about which tools to use
 and in what order, facilitating autonomous problem-solving. It also offers modules for "Retrieval," integrating external knowledge bases like vector stores for custom data, and "Memory," crucial for maintaining conversational context.

## Batch (These are for non realtime scenarios, where response times are higher so Provides give huge discounts)
- Batching a collection of independent requests to a model can significantly improve the performance and reduce the cost, as the processing can be done in parallel
- Batches will be executed in the queue when ever resources are available, they are given less priority so cost is less

In [18]:
responses = model.batch(
    [
        "What is 2+2=?",
        "What is Loop Engineering in Agentic AI?",
        "What are the Advantages of Batching in LangChain 'model.batch()'"
    ],
    config={
        'max_concurrency': 5 # so at a time it can process maximum of 5 calls in parallel
    }
)
for response in responses:
    print(response)

content='2 + 2 = 4' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f0dc7-ce30-79d0-9063-a1abb05ea287-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 32, 'total_tokens': 40, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 25}}
content='In the context of Agentic AI, **Loop Engineering** refers to the deliberate design, implementation, and optimization of the continuous feedback and action cycles that enable AI agents to operate autonomously, adapt to changing environments, learn from their experiences, and progressively achieve complex goals.\n\nEssentially, it\'s about building robust and intelligent **Perceive-Plan-Act-Evaluate-Reflect** loops (or similar iterative cycles) that allow an AI agent to:\n\n1.  **Perceive:** Gather information from its environment (e.g., read a document, check an

# Summary
- We have seen how to interact with different LLM Providers like OpenAI, Gemini, GROQ, OLLAMA (Local models)
- We have seen different ways of interacting with the LLMs using LangChain library
    - Universal Approach: `init_chat_model` approach where we can mention any model as the parameter from any provider
    - Provider Specific Approach: ChatOllama, ChatOpenAi, ChatGoogleGenerativeAI, ChatGroq from their respective langchain flavoured libraries
- Streaming vs Batch Processing, where streaming can improve the user experience as it shows the output response chunk by chunk in progressive way, Batch Processing will reduce the cost and increase the speed with concurrency where it can hit LLM with multiple queries at a time in parallel.